This notebook processes the two replicates of the cohen retina scMPRA dataset into an ortho object.

# Setup

In [1]:
import scMPRAforge as scm

2025-10-27 10:46:49.729841: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-27 10:46:50.631050: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
#create dask cluster

from dask_jobqueue import SLURMCluster
from dask.distributed import Client

cluster=SLURMCluster(
    cores=4,#cores per slurm job
    memory="64G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=9:00:00",
        f"--output=slave_%j.out"]
)

cluster.scale(jobs=6)

client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="30s"  # Worker heartbeat interval
    )

#from dask.distributed import Client, LocalCluster
#cluster=LocalCluster(memory_limit='8GB')
#client = Client(cluster)

# Describe with an ortho

Regarding reference cell-type,
> Reproducibility was highest in rod cells (Spearman’s ρ = 0.97, Pearson’s R = 0.98) because rod cells are the most abundant cell type in the mouse retina.

Regarding negative controls 
> In this experiment, we define the effect of a mutation as its relative fold-change to the WT Gnb3 promoter in each cell type because the Gnb3 promoter is expressed at different levels across cell types (Fig. 5a).

Interestingly, there are two wt promoters. Some code in the cohen paper's zenodo (`Part2_section6_retina_analysis.ipynb`) suggests that they should be treated together:

```
rep1_wt_df = rep1_exp.loc[rep1_exp['name'].str.contains('wt')]
mean_exp = np.mean(rep1_wt_df['mean'])
rep1_exp['activity'] = rep1_exp['mean']/mean_exp
```

So we will simply lump both, despite the fact that they have different sequences.

In [4]:
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
path="/gpfs/gibbs/pi/reilly/tabula_data/cohen"
name="ortho_primordial"


import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
else:
    print("[+] Model not found. Creating...")

    #load data
    cohen=scm.scMPRA_data.from_parquet(f"{path}/retina_single_counting_u6.scmpra")
    cohen.set_negative_controls(["wt_1","wt_2"])
    cohen.set_reference_cell("Rod")
    cohen.ortho_filter()

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=cohen)
    primordial.extract_params(client)
    #primordial.save(path,name)


[+] Model not found. Creating...


scMPRAforge: INFO: Dropped 0 of 456 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [22]:
primordial.by_cell_type.model

{'reference': <Future: pending, key: _label_tensorzinb_regressors-ead2c04ef298f3d8b080b81aede564fb>,
 'Mueller Glia': <Future: finished, type: dict, key: _label_tensorzinb_regressors-2d45a71ff5cb499d42d056ee8b9b9a14>,
 'Interneuron': <Future: finished, type: dict, key: _label_tensorzinb_regressors-1c7cfff3fe4684d835f0e40dd74b5305>,
 'Bipolar': <Future: finished, type: dict, key: _label_tensorzinb_regressors-7673c8dc48010fec914f55060c2eee8f>}

In [20]:
client.who_has(primordial.by_cell_type.model['Interneuron'])

Key,Copies,Workers
_label_tensorzinb_regressors-1c7cfff3fe4684d835f0e40dd74b5305,1,tcp://10.178.138.27:41829


In [24]:
client.who_has(primordial.by_cell_type.model['reference'])

Key,Copies,Workers
_label_tensorzinb_regressors-ead2c04ef298f3d8b080b81aede564fb,0,


In [10]:
primordial.by_cre.model

{'combo_Mute_crx1': <Future: finished, type: dict, key: _label_tensorzinb_regressors-168e929059bcf8dd4c10516ffb33c1f7>,
 'combo_Mute_crx2': <Future: finished, type: dict, key: _label_tensorzinb_regressors-6f439db62295d6d6631ec2b3fd8516ad>,
 'combo_Mute_crx2_crx1': <Future: finished, type: dict, key: _label_tensorzinb_regressors-897748f52323de0cf9532d4bdba696ce>,
 'combo_Mute_crx3_crx1': <Future: finished, type: dict, key: _label_tensorzinb_regressors-a10afff1531df06bc240bab1578021d5>,
 'combo_Mute_crx4': <Future: finished, type: dict, key: _label_tensorzinb_regressors-bc2b09862e4417c893b0b5b72a20b411>,
 'combo_Mute_crx4_crx1': <Future: finished, type: dict, key: _label_tensorzinb_regressors-3f09a337bfaafb424390317c0cdabe76>,
 'combo_Mute_crx4_crx2': <Future: finished, type: dict, key: _label_tensorzinb_regressors-7ebd268a37de5c84dd03f3bebcaabba1>,
 'combo_Mute_crx5': <Future: finished, type: dict, key: _label_tensorzinb_regressors-9494aae52b65104451d5030b291a3825>,
 'combo_Mute_crx5_cr

In [ ]:
cluster.close()
client.close()

Examining QC metrics : mean VS estimated parameter